# 15.2n - Binary gingival YOLOv8n

Controlled baseline. Source-annotated gingival inflammation localization. Read dataset limitations in notebook 15.1. Training execution is a separate stage.

## 1. Install the pinned environment

In [ ]:
%pip install -q ultralytics==8.3.228

## 2. Verify prepared input and GPU
Attach the published binary dataset after its audit. Require exactly one dataset YAML.

In [ ]:
from pathlib import Path
import torch, json, hashlib, zipfile, time
from ultralytics import YOLO
import pandas as pd
import matplotlib.pyplot as plt
# Kaggle expands uploaded ZIP datasets into mounted files.
import shutil
manifests=list(Path('/kaggle/input').rglob('manifest.csv'))
assert len(manifests)==1, manifests
source=manifests[0].parent
target=Path('/tmp/gum_binary_v1')
shutil.copytree(source,target,dirs_exist_ok=True)
assert hashlib.sha256((target/'manifest.csv').read_bytes()).hexdigest()=='5c2b9ceb917e88d2ebff07719105b01dd9c1c46076bca2671879cc39475776f0'
checksums=json.loads((target/'checksums.json').read_text())
for name, expected in checksums.items():
    assert hashlib.sha256((target/name).read_bytes()).hexdigest()==expected, name
assert len(checksums)==1814
(target/'data.yaml').write_text(f'path: {target}\ntrain: images/train\nval: images/val\nnames:\n  0: visible_gingival_inflammation\n')
data=target/'data.yaml'
Path('/kaggle/working/input_manifest.csv').write_bytes((target/'manifest.csv').read_bytes())
print('Manifest SHA256:','5c2b9ceb917e88d2ebff07719105b01dd9c1c46076bca2671879cc39475776f0')
display(pd.read_csv(target/'manifest.csv').groupby('split').agg(images=('sample_id','size'), boxes=('positives','sum')))
assert torch.cuda.is_available()
print('Visible GPUs:',[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

## Inspect annotations and letterboxing
Show a seeded training image with target boxes, then the 640 aspect-preserving input. YOLO converts input pixels to floating-point values in [0,1]. Input staging verified every image and label checksum.

In [ ]:
from PIL import Image, ImageDraw
from ultralytics.data.augment import LetterBox
import numpy as np
path=sorted((target/'images/train').iterdir())[0]
im=Image.open(path).convert('RGB');draw=ImageDraw.Draw(im)
for line in (target/'labels/train'/(path.stem+'.txt')).read_text().splitlines():
    _,x,y,w,h=map(float,line.split())
    draw.rectangle(((x-w/2)*im.width,(y-h/2)*im.height,(x+w/2)*im.width,(y+h/2)*im.height),outline='lime',width=2)
fig,axes=plt.subplots(1,2,figsize=(12,5))
axes[0].imshow(im);axes[0].set_title('Source annotation')
axes[1].imshow(LetterBox((640,640))(image=np.asarray(im)));axes[1].set_title('640 letterbox geometry')
for ax in axes: ax.axis('off')
plt.tight_layout();plt.show()

## 3. Train the controlled model
Both scales use 640 letterboxing, pixel normalization, the same seed and optimizer. Restrained color augmentation avoids large perturbations of gingival color. Use 60 fixed epochs and periodic checkpoints. A five-hour epoch-boundary guard requests a normal stop. Periodic checkpoints retain resumable training state; collect them before session expiry.

In [ ]:
model=YOLO('yolov8n.pt')
started=time.monotonic()
def time_guard(trainer):
    if time.monotonic()-started > 5*3600:
        trainer.stop=True
        print('Five-hour guard reached; retain periodic checkpoints for continuation')
model.add_callback('on_train_epoch_end',time_guard)
results=model.train(data=str(data),epochs=60,patience=0,imgsz=640,batch=16,
    device=0,workers=2,seed=42,deterministic=True,optimizer='AdamW',lr0=0.001,
    weight_decay=0.0005,hsv_h=0.0,hsv_s=0.2,hsv_v=0.2,fliplr=0.5,
    flipud=0.0,mosaic=1.0,close_mosaic=10,mixup=0.0,
    project='/kaggle/working/runs',name='gum_binary_v8n',
    save=True,save_period=10,plots=True)
print(results.results_dict)

## 4. Inspect learning curves
Report detection AP and recall. These metrics address a different task from the earlier three-grade crop classifier.

In [ ]:
run=Path(results.save_dir)
history=pd.read_csv(run/'results.csv'); history.columns=history.columns.str.strip()
display(history.tail())
history.plot(x='epoch',y=['metrics/mAP50(B)','metrics/mAP50-95(B)'],grid=True)
plt.show()
(run/'metrics.json').write_text(json.dumps(results.results_dict,indent=2))
print('Best checkpoint SHA256',hashlib.sha256((run/'weights/best.pt').read_bytes()).hexdigest())